# 02 — Pré-processamento e Feature EngineeringCobre a **Etapa 3** do enunciado: dados faltantes, padronização e criação de novas features.> Responsável: _(Pré-processamento & Features)_

In [ ]:
import syssys.path.append("..")import matplotlib.pyplot as pltimport numpy as npimport pandas as pdimport seaborn as snsfrom src import data_loader as dlfrom src import preprocessing as pppd.set_option("display.max_columns", None)sns.set_theme(style="whitegrid")plt.rcParams["figure.figsize"] = (9, 5)

In [ ]:
df = pd.read_csv("../data/processed/wine_com_alvo.csv")X, y = dl.get_features_and_target(df)print("X:", X.shape, "| y:", y.shape)X.head()

> ⚠️ `get_features_and_target` remove a coluna `quality` original de propósito.> Mantê-la seria **vazamento de dados**: o alvo binário foi derivado dela, e qualquer modelo> acertaria 100% sem aprender nada.

## 3.1 Tratamento de dados faltantes

In [ ]:
display(pp.missing_report(X))# Se houver faltantes, imputar pela mediana (robusta a outliers):# X = X.fillna(X.median())# Duplicadosprint("Duplicados:", pp.duplicates_report(df))# df = df.drop_duplicates()

## 3.2 Feature engineeringTrês features derivadas, cada uma com leitura enológica (ver `src/preprocessing.py`):- `free_to_total_so2` — proporção de SO2 livre; é o livre que efetivamente protege o vinho da oxidação.- `acidity_ratio` — acidez fixa ÷ volátil; separa acidez "boa" (frescor) de acidez "ruim" (defeito).- `alcohol_density` — combina teor alcoólico e densidade, que são fisicamente ligados.> Se decidirem **não** usar alguma, registre o motivo — o enunciado pede feature engineering> "se considerado relevante", então a justificativa vale tanto quanto a feature.

In [ ]:
X_fe = pp.add_engineered_features(X)X_fe[["free_to_total_so2", "acidity_ratio", "alcohol_density"]].describe().T

In [ ]:
# As novas features realmente separam as classes?fig, axes = plt.subplots(1, 3, figsize=(15, 4))for ax, col in zip(axes, ["free_to_total_so2", "acidity_ratio", "alcohol_density"]):    sns.boxplot(x=y, y=X_fe[col], palette=["#9CA3AF", "#7C3A4E"], ax=ax)    ax.set_xticklabels(["Baixa/Média", "Alta"])    ax.set_xlabel("")plt.suptitle("Features criadas x classe de qualidade")plt.tight_layout()plt.savefig("../results/figures/features_criadas.png", dpi=150, bbox_inches="tight")plt.show()

## 3.3 Divisão treino/testeSplit **estratificado** — obrigatório com classes desbalanceadas, para que treino e testemantenham a mesma proporção de vinhos de alta qualidade.

In [ ]:
X_train, X_test, y_train, y_test = pp.split_data(X_fe, y, test_size=0.2)print("Treino:", X_train.shape, "| Teste:", X_test.shape)print("\nProporção da classe 1 — treino:", round(y_train.mean() * 100, 2), "%")print("Proporção da classe 1 — teste: ", round(y_test.mean() * 100, 2), "%")

## 3.4 Padronização`StandardScaler` ajustado **somente no treino** e aplicado no teste. Ajustar no conjuntocompleto vazaria informação do teste para o treino.

In [ ]:
X_train_sc, X_test_sc, scaler = pp.scale_data(X_train, X_test)X_train_sc.describe().T[["mean", "std"]].round(3)

In [ ]:
# Persistir para o notebook 03X_train_sc.to_csv("../data/processed/X_train.csv", index=False)X_test_sc.to_csv("../data/processed/X_test.csv", index=False)y_train.to_csv("../data/processed/y_train.csv", index=False)y_test.to_csv("../data/processed/y_test.csv", index=False)print("Bases de treino e teste salvas em data/processed/")